# Prompt Engineering — Live Demo

This notebook follows the module: prompt anatomy, templates, in-context learning, refinement, structured JSON, task patterns, and multi-turn APIs.

Before running: add `GROQ_API_KEY=your_key_here` to the project's `.env` file.

In [ ]:
%pip install -q groq python-dotenv

import json
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
api_key = os.getenv('GROQ_API_KEY')
if not api_key:
    raise RuntimeError('GROQ_API_KEY is missing. Add it to .env before continuing.')

client = Groq(api_key=api_key)
MODEL = 'llama-3.3-70b-versatile'

def ask(prompt, system='You are a helpful assistant.', temperature=0.2, max_tokens=300):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()

print(f'Ready. Model: {MODEL}')

## 1. Prompt refinement: vague → specific

Compare the response quality after adding role, audience, length, tone, and output constraints.

In [ ]:
bad_prompt = 'Write about photosynthesis.'
good_prompt = '''Role: You are an experienced high-school biology teacher.
Context: A Class 10 student is preparing for an exam.
Task: Explain photosynthesis.
Constraints: Use simple language, maximum 120 words, no unexplained jargon.
Output format: Exactly 3 bullet points followed by a one-line summary.'''.strip()

print('VAGUE PROMPT:\n', ask(bad_prompt, temperature=0.7))
print('\n' + '=' * 70 + '\n')
print('REFINED PROMPT:\n', ask(good_prompt, temperature=0.2))

## 2. Prompt template

A template separates fixed instructions from changing input data—the same idea used in production chatbots and RAG systems.

In [ ]:
SUMMARY_TEMPLATE = '''You are a professional editor.
Summarize the following {document_type} in exactly {num_sentences} sentences.
Focus on: {focus_area}.

Text:
{input_text}'''.strip()

paper_text = '''A study of 500 first-year students found that weekly retrieval-practice quizzes
improved final-exam scores by 12% compared with rereading notes. The benefit was strongest
for students who reviewed feedback within 24 hours.'''

prompt = SUMMARY_TEMPLATE.format(
    document_type='research abstract',
    num_sentences=2,
    focus_area='method and key finding',
    input_text=paper_text,
)
print(ask(prompt))

## 3. Zero-shot vs. few-shot classification

Few-shot examples show the required pattern directly. Change the final review and run again.

In [ ]:
review = 'The speakers were good, but registration took far too long.'

zero_shot = f'''Classify this college-fest review as Positive, Negative, or Neutral.
Respond with the label only.
Review: {review}'''

few_shot = f'''Classify each review as Positive, Negative, or Neutral. Respond with the label only.
Review: Amazing performances and friendly volunteers! -> Positive
Review: It was okay; nothing stood out. -> Neutral
Review: The event started two hours late. -> Negative
Review: {review} ->'''

print('Zero-shot:', ask(zero_shot, max_tokens=10))
print('Few-shot: ', ask(few_shot, max_tokens=10))

## 4. Structured output: machine-parseable JSON

JSON mode helps ensure valid JSON; application code must still validate the expected keys and types.

In [ ]:
text = 'Aarav Mehta is 19 years old, studies in Pune, and is interested in robotics and debate.'
json_prompt = f'''Extract student information from the text below.
Return JSON with this exact schema and no additional keys:
{{"name": "string", "age": integer, "city": "string", "interests": ["string"]}}
Text: {text}'''

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': json_prompt}],
    response_format={'type': 'json_object'},
    temperature=0,
)
data = json.loads(response.choices[0].message.content)

required = {'name': str, 'age': int, 'city': str, 'interests': list}
assert set(data) == set(required), 'Unexpected JSON keys'
assert all(isinstance(data[key], type_) for key, type_ in required.items()), 'Wrong JSON value type'
data

## 5. Six task-based prompt patterns

This reusable dictionary covers summarization, question answering, extraction, classification, translation, and code generation. Select any task and edit its input.

In [ ]:
TASK_PROMPTS = {
    'summarization': 'Summarize in 2 sentences, focusing on key findings. Text: {text}',
    'question_answering': 'Answer using only this context. If absent, say Not found. Context: {context} Question: {question}',
    'extraction': 'Extract all {entity_type} as a JSON list. Text: {text}',
    'classification': 'Classify the text as {labels}. Respond with one label only. Text: {text}',
    'translation': 'Translate from {source_lang} to {target_lang}; preserve tone. Text: {text}',
    'code_generation': 'Write a {language} function for: {task}. Include a docstring and one example.',
}

task = 'code_generation'  # Try any key above
prompt = TASK_PROMPTS[task].format(language='Python', task='return the largest number in a list')
print(ask(prompt, max_tokens=250))

## 6. Multi-turn conversation and API controls

The history list is the model's session memory. Each turn resends prior messages; `temperature=0` favors repeatability while larger values add variety.

In [ ]:
history = [
    {'role': 'system', 'content': 'You are a patient calculus tutor. Use clear, short explanations.'},
    {'role': 'user', 'content': 'Explain gradient descent in two sentences.'},
]

first = client.chat.completions.create(model=MODEL, messages=history, temperature=0.2, max_tokens=120)
first_text = first.choices[0].message.content
print('Turn 1:', first_text)

history += [
    {'role': 'assistant', 'content': first_text},
    {'role': 'user', 'content': 'Now explain the same idea as if I am 10 years old, using an analogy.'},
]
second = client.chat.completions.create(model=MODEL, messages=history, temperature=0.5, max_tokens=150)
print('\nTurn 2:', second.choices[0].message.content)